In [51]:

import re
import spacy
import html
import os
import random
import json

def split_sentences(nlp, text):
    """
    Splits the given text into sentences using spaCy's sentence segmenter.
    """
    doc = nlp(text)
    return [sent.text for sent in doc.sents]

def read_and_process_json_sections(json_file, nlp):
    global skipped_file
    """
    Reads a JSON file, processes sections into sentences, and maps mentions to sentences.
    """
    with open(json_file, "r", encoding="utf-8") as file:
        data = json.load(file)

    # Process each page in the JSON data
    final_data = []
    for page_id, page_content in enumerate(data['pages']):
        print(f"\033[91mProcessing\033[0m Page ID: {page_id}")

        # Clean the content by replacing </p> with a space and removing <p> tags
        cleaned_content = re.sub(r'<\/?p>', '', page_content)

        # Extract entities from annotations
        entities = []
        for annotation in data['annotations']:
            if annotation['relationships']['date']['page']== page_id:
                start = annotation['textContext']['start']
                end = annotation['textContext']['end']
                entity_type = annotation['label']
                mention_text = cleaned_content[start:end]
                # Append the entity tuple
                entities.append({
                    "start": start,
                    "end": end,
                    "type": entity_type,
                    "text": annotation['textContext']['text']
                })
                

        # Order entities based on the value of start
        entities = sorted(entities, key=lambda x: x['start'])
        print(f"Entities: {entities}")

        # Split the section into sentences
        sentences = split_sentences(nlp, cleaned_content)

        sentence_start = 0
        entity_index = 0  # Index to track the current entity being processed
        num_entities = len(entities)

        for sentence in sentences:
            sentence_end = sentence_start + len(sentence)
            sentence_entities = []

            # Process entities that belong to the current sentence
            while entity_index < num_entities and entities[entity_index]["start"] >= sentence_start and entities[entity_index]["end"] <= sentence_end:
                entity = entities[entity_index]
                # print(f"Entity: {entity}")
                # Calculate relative start and end positions in the sentence
                relative_start = entity["start"] - sentence_start
                relative_end = entity["end"] - sentence_start
                entity_text = entity["text"]
                start = sentence.find(entity_text)

                # Validate that the extracted text matches the entity text
                if start != -1:
                    end = start + len(entity_text)
                    entity_type = entity["type"].lower()
                    global unique_entity_types
                    global entity_type_counts
                    unique_entity_types.add(entity_type)
                    if entity_type in entity_type_counts:
                        entity_type_counts[entity_type] += 1
                    else:
                        entity_type_counts[entity_type] = 1
                    # if entity["type"] != "R/O" and entity["type"] != "sDx" and entity["type"] != "SYM" and entity["type"] != "Dx":
                    #     if entity["type"] in ["sDrug","cDrug", "drug"]:
                    #         sentence_entities.append((start, end, "Drug"))
                    #         print(f"\033[91mchanged to Drug: {entity['type']} at ({start}, {end}), and the word is {entity_text}\033[0m")
                    #     elif entity["type"] == "mAE" or entity["type"] == "ae":
                    #         sentence_entities.append((start, end, "AE"))
                    #         print(f"\033[91mchanged to AE: {entity['type']} at ({start}, {end}, and the word is {entity_text})\033[0m")
                    #     else:
                    #         sentence_entities.append((start, end, entity_type))
                    # else:
                    #     # print(f"\033[93mskip these entities: {entity['type']} at ({start}, {end})\033[0m")
                    #     skipped_file += 1
                        # print(f"skip these entities: {entity["type"]} at ({start}, {end})")
                    # sentence_entities.append((start, end, entity_type))
                    # sentence_entities.append((start, end, entity_type, entity_text))
                    # print(f"Entity matched: {entity_text} at ({start}, {end})")
                else:
                    print(
                        f"\033[93mWarning: Text mismatch for entity '{entity_text}' in {sentence} "
                        f"at indices ({relative_start}, {relative_end}).\033[0m" 
                    )

                # Move to the next entity
                entity_index += 1

            # Add the sentence and its entities to the labeled data
            if sentence_entities:
                final_data.append((sentence, {"entities": sentence_entities}))
                # print(f"\033[92mmatched Sentence: {sentence}\033[0m")
                # print(f"\033[92mmatched Entities: {sentence_entities}\033[0m")

            # Update the sentence_start for the next sentence
            if sentence_start + len(sentence) < len(cleaned_content) and cleaned_content[sentence_start + len(sentence)] == ' ':
                sentence_start = sentence_end + 1  # Account for the space/newline between sentences
            else:
                sentence_start = sentence_end

    return final_data

def get_file_list(folder_path):
    """
    Get a list of all JSON files in a folder.
    """
    return [os.path.join(folder_path, file) for file in os.listdir(folder_path) if file.endswith(".json")]


def split_file_list(file_list, train_ratio=0.8):
    """
    Split the list of files into train and test datasets.
    """
    random.seed(123)
    random.shuffle(file_list)
    split_idx = int(len(file_list) * train_ratio)
    return file_list[:split_idx], file_list[split_idx:]

def process_files(file_list, nlp):
    """
    Process JSON files into spaCy-compatible training data.
    """
    processed_data = []
    for json_file in file_list:
        print(f"\033[91mProcessing\033[0m file: {json_file}")
        processed_data.extend(read_and_process_json_sections(json_file, nlp))
    return processed_data
unique_entity_types = set()
entity_type_counts = {}
# Example usage
folder_path = 'C:/Users/Wenjuan.Zhang/OneDrive - FDA/Documents/VAERS_LLM/'  # Replace with your folder path
file_list = get_file_list(folder_path)

# Split into train and test datasets
train_files, test_files = split_file_list(file_list)

# Load a spaCy language model for sentence splitting
nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")  # Add sentence segmenter
skipped_file = 0
# Process train and test files
train_data = process_files(train_files, nlp)
test_data = process_files(test_files, nlp)
print(test_files)
print(f"Skipped files: {skipped_file}")
# Output a sample from the formatted data
for sample in train_data[:3]:  # Display first 3 samples
    print(sample)
# Sort and print all unique entity types
sorted_unique_entity_types = sorted(unique_entity_types)
print(f"\033[92mUnique Entity Types: {sorted_unique_entity_types}\033[0m")
sorted_entity_type_counts = sorted(entity_type_counts.items())

# Print each key-value pair on a new line, separated by commas
for entity_type, count in sorted_entity_type_counts:
    print(f"{entity_type}: {count}")



Processing file: C:/Users/Wenjuan.Zhang/OneDrive - FDA/Documents/VAERS_LLM/VAERS_269542-1.json
Processing Page ID: 0
Entities: [{'start': 71, 'end': 96, 'type': 'SYM', 'text': 'injection site cellulitis'}, {'start': 71, 'end': 96, 'type': 'sym', 'text': 'injection site cellulitis'}, {'start': 102, 'end': 104, 'type': 'age', 'text': '35'}, {'start': 114, 'end': 120, 'type': 'sex', 'text': 'female'}, {'start': 153, 'end': 176, 'type': 'VAX', 'text': 'influenza virus vaccine'}, {'start': 153, 'end': 186, 'type': 'vax', 'text': 'influenza virus vaccine (Fluarix)'}, {'start': 178, 'end': 185, 'type': 'VAX', 'text': 'Fluarix'}, {'start': 217, 'end': 262, 'type': 'VAX', 'text': '23-valent pneumococcal polysaccharide vaccine'}, {'start': 217, 'end': 274, 'type': 'vax', 'text': '23-valent pneumococcal polysaccharide vaccine (Pneumovax)'}, {'start': 264, 'end': 273, 'type': 'VAX', 'text': 'Pneumovax'}, {'start': 317, 'end': 327, 'type': 'MHx', 'text': 'cellulitis'}, {'start': 317, 'end': 327, 't

In [52]:

# Filter the list to only keep strings that contain uppercase letters
filtered_entity_types = [entity for entity in unique_entity_types if any(char.isupper() for char in entity)]
entity_type_mapping = {entity.lower(): entity for entity in unique_entity_types}
# Print the filtered list
print(filtered_entity_types)
# List to be changed
to_change = ["ae", "drug"]

# Change the strings to the one with uppercase letters according to filtered_entity_types
changed_list = [entity for entity in filtered_entity_types if entity.lower() in to_change]
print(changed_list)


filtered_entity_types_lower = [entity.lower() for entity in filtered_entity_types]

[]
[]


In [53]:
# removed_entity = ["SYM","ae-infectious","ae=infection<","ae\metabolism","bsym.sleeping","drug)lanuts<",
#                   "drug)methadone<","drug)morphine","drug)morphine<","lab=inflammatory","p=literature",
#                   "sDx"]

entities_with_low_count = [entity_type for entity_type, count in entity_type_counts.items() if count < 10]

# Print the list of entities with count < 10
print(entities_with_low_count)

[]


In [54]:
import re
import spacy
import html
import os
import random
import json

def split_sentences(nlp, text):
    doc = nlp(text)
    return [sent.text for sent in doc.sents]

def read_and_process_json_sections(json_file, nlp, file_type):
    global skipped_file
    with open(json_file, "r", encoding="utf-8") as file:
        data = json.load(file)

    final_data = []

    for page_id, page_content in enumerate(data['pages']):
        # Replace <p>, </p> and \n with spaces
        cleaned_content = re.sub(r'</?p>', ' ', page_content)
        cleaned_content = re.sub(r'\n', ' ', cleaned_content)

        # Extract and filter entities
        entities = []
        for annotation in data['annotations']:
            if annotation['relationships']['date']['page'] != page_id:
                continue
            if annotation['note'] not in file_type:
                continue

            start = annotation['textContext']['start']
            end = annotation['textContext']['end']
            entity_type = annotation['label']
            mention_text = cleaned_content[start:end]

            if annotation['note'] in file_type and annotation['label'] != "CoD" and annotation['label'] != "cod":
                    # Append the entity tuple
                    entities.append({
                        "start": start,
                        "end": end,
                        "type": entity_type,
                        "text": annotation['textContext']['text']
                    })

        # Sort entities by starting position
        entities.sort(key=lambda x: x['start'])

        sentences = split_sentences(nlp, cleaned_content)
        sentence_start = 0
        entity_index = 0
        num_entities = len(entities)

        for sentence in sentences:
            sentence_end = sentence_start + len(sentence)
            sentence_entities = []

            while (entity_index < num_entities and
                   sentence_start <= entities[entity_index]["start"] < sentence_end and
                   sentence_start <= entities[entity_index]["end"] <= sentence_end):

                entity = entities[entity_index]
                entity_text = entity["text"]
                relative_start = entity["start"] - sentence_start
                relative_end = entity["end"] - sentence_start

                # Find the actual position of entity text in the sentence
                match_start = sentence.find(entity_text)
                if match_start != -1:
                    match_end = match_start + len(entity_text)
                    ent_type = entity["type"]

                    # Standardize case and match entity types
                    if ent_type.islower() and ent_type in filtered_entity_types_lower:
                        matched_type = [t for t in filtered_entity_types if t.lower() == ent_type][0]
                        print(f"\033[91mChanged to {matched_type}: {ent_type} at ({match_start}, {match_end}) with word '{entity_text}'\033[0m")
                        ent_type = matched_type

                    if ent_type not in entities_with_low_count:
                        unique_entity_types.add(ent_type)
                        entity_type_counts.setdefault(ent_type, 0)
                        entity_type_counts[ent_type] += 1

                        sentence_entities.append((
                            match_start, match_end, ent_type
                        ))
                    else:
                        print(f"\033[91mLow-count entity: {entity['type']} at ({match_start}, {match_end})\033[0m")
                else:
                    print(f"\033[93mWarning: Text mismatch for entity '{entity_text}' in sentence: \"{sentence}\" at ({relative_start}, {relative_end})\033[0m")

                entity_index += 1

            if sentence_entities:
                final_data.append((sentence, {"entities": sentence_entities}))

            # Update sentence start position for next iteration
            if sentence_start + len(sentence) < len(cleaned_content) and cleaned_content[sentence_start + len(sentence)] == ' ':
                sentence_start = sentence_end + 1
            else:
                sentence_start = sentence_end

    return final_data
def process_files(file_list, nlp, file_type):
    """
    Process JSON files into spaCy-compatible training data.
    """
    processed_data = []
    for json_file in file_list:
        # print(f"\033[91mProcessing\033[0m file: {json_file}")
        processed_data.extend(read_and_process_json_sections(json_file, nlp, file_type))
    return processed_data


In [55]:
###train data with SME1 and test data with SME1, too
train_data = process_files(train_files, nlp, ["LLM"])  #"LLM","SME1"
test_data = process_files(test_files, nlp, ["LLM"])
print(test_files)
print(f"Skipped files: {skipped_file}")
# Output a sample from the formatted data
for sample in train_data[:3]:  # Display first 3 samples
    print(sample)

['C:/Users/Wenjuan.Zhang/OneDrive - FDA/Documents/VAERS_LLM/VAERS_253926-1.json', 'C:/Users/Wenjuan.Zhang/OneDrive - FDA/Documents/VAERS_LLM/VAERS_517680-1.json', 'C:/Users/Wenjuan.Zhang/OneDrive - FDA/Documents/VAERS_LLM/VAERS_530544-1.json', 'C:/Users/Wenjuan.Zhang/OneDrive - FDA/Documents/VAERS_LLM/VAERS_193281-1.json', 'C:/Users/Wenjuan.Zhang/OneDrive - FDA/Documents/VAERS_LLM/VAERS_640405-1.json', 'C:/Users/Wenjuan.Zhang/OneDrive - FDA/Documents/VAERS_LLM/VAERS_673931-1.json', 'C:/Users/Wenjuan.Zhang/OneDrive - FDA/Documents/VAERS_LLM/VAERS_498088-1.json', 'C:/Users/Wenjuan.Zhang/OneDrive - FDA/Documents/VAERS_LLM/VAERS_638868-1.json', 'C:/Users/Wenjuan.Zhang/OneDrive - FDA/Documents/VAERS_LLM/VAERS_174771-1.json', 'C:/Users/Wenjuan.Zhang/OneDrive - FDA/Documents/VAERS_LLM/VAERS_364032-1.json', 'C:/Users/Wenjuan.Zhang/OneDrive - FDA/Documents/VAERS_LLM/VAERS_398125-1.json', 'C:/Users/Wenjuan.Zhang/OneDrive - FDA/Documents/VAERS_LLM/VAERS_635595-1.json', 'C:/Users/Wenjuan.Zhang/One

In [56]:
print(len(train_files))
print(len(test_files))

800
200


In [57]:

skipped_count = 0
remained_count = 0
exceedlength = 0
import warnings
from pathlib import Path
import spacy
from spacy.tokens import DocBin
from spacy.util import filter_spans
import re
from collections import defaultdict

def validate_entities(entities):
    """
    Validate that there are no overlapping entities.
    Args:
        entities (list): List of tuples (start, end, label).
    Returns:
        bool: True if no overlaps, False otherwise.
    """
    entities = sorted(entities, key=lambda x: x[0])  # Sort by start index
    for i in range(len(entities) - 1):
        if entities[i][1] > entities[i + 1][0]:  # Check for overlap
            return False
    return True

def clean_entities(entities, text):
    """
    Clean entity spans by removing leading and trailing whitespace.
    Args:
        entities (list): List of tuples (start, end, label).
        text (str): The text containing the entities.
    Returns:
        list: Cleaned list of entities.
    """
    cleaned_entities = []
    for start, end, label in entities:
        # Ensure start and end are within bounds
        start = max(0, min(start, len(text)))
        end = max(0, min(end, len(text)))
        
        # Remove leading whitespace
        while start < end and text[start].isspace():
            start += 1
        # Remove trailing whitespace
        while end > start and text[end - 1].isspace():
            end -= 1
        if start < end:
            cleaned_entities.append((start, end, label))
    return cleaned_entities

def split_sentences(nlp, text, entities):
    """
    Splits the given text into sentences using spaCy's sentence segmenter.
    If a sentence is longer than 512 characters, it is split at the closest whitespace to the left of the 512th character.
    Adjusts entity positions accordingly.
    """
    doc = nlp(text)
    sentences = [sent.text for sent in doc.sents]
    split_sentences = []
    split_entities = []

    for sentence in sentences:
        while len(sentence) > 512:
            # Find the closest whitespace to the left of the 512th character
            split_index = sentence.rfind(' ', 0, 512)
            if split_index == -1:
                split_index = 512  # If no whitespace is found, split at 512

            # Split the sentence at the whitespace
            part1 = sentence[:split_index].strip()
            part2 = sentence[split_index:].strip()
            split_sentences.append(part1)

            # Adjust entity positions
            part1_len = len(part1)
            part1_entities = []
            part2_entities = []
            for start, end, label in entities:
                if start < part1_len:
                    if end <= part1_len:
                        part1_entities.append((start, end, label))
                    else:
                        part1_entities.append((start, part1_len, label))
                        part2_entities.append((0, end - part1_len, label))
                else:
                    part2_entities.append((start - part1_len, end - part1_len, label))
            split_entities.append(part1_entities)
            sentence = part2
            entities = part2_entities

        split_sentences.append(sentence)
        split_entities.append(entities)

    return split_sentences, split_entities

def truncate_text_and_entities(text, entities, max_length=512):
    """
    Truncate text and entities to a maximum length.
    Args:
        text (str): The text to be truncated.
        entities (list): List of tuples (start, end, label).
        max_length (int): Maximum length of the text.
    Returns:
        tuple: Truncated text and entities.
    """
    global exceedlength
    if len(text) <= max_length:
        return text, entities

    truncated_text = text[:max_length]
    truncated_entities = []
    for start, end, label in entities:
        if start < max_length:
            if end > max_length:
                exceedlength += 1
                end = max_length
            truncated_entities.append((start, end, label))
    return truncated_text, truncated_entities

def convert(lang: str, TRAIN_DATA, output_path: Path):
    """
    Convert training data into SpaCy's .spacy format.
    Args:
        lang (str): Language code (e.g., "en").
        TRAIN_DATA (list): List of training data in the format [(text, {"entities": [...]})].
        output_path (Path): Path to save the .spacy file.
    """
    global skipped_count
    global remained_count
    nlp = spacy.blank(lang)  # Create a blank language model
    nlp.add_pipe("sentencizer")  # Add the sentencizer component to the pipeline
    db = DocBin()  # Initialize DocBin to store Doc objects
    entity_counts = defaultdict(int)
    for text, annot in TRAIN_DATA:
        # Split sentences and adjust entities
        sentences, entities_list = split_sentences(nlp, text, annot["entities"])
        for sentence, entities in zip(sentences, entities_list):
            # Truncate text and entities to the maximum length
            sentence, entities = truncate_text_and_entities(sentence, entities, max_length=512)
            doc = nlp.make_doc(sentence)  # Create a new Doc object
            
            # Validate entities for overlaps
            if not validate_entities(entities):
                skipped_count += 1
                warnings.warn(f"Skipping text with overlapping entities: {repr(sentence)}")
                continue
            
            # Clean entity spans
            entities = clean_entities(entities, sentence)
            
            ents = []
            for start, end, label in entities:
                span = doc.char_span(start, end, label=label, alignment_mode="strict")  #strict
                if span is None:
                    skipped_count += 1
                    msg = (f"Skipping entity [{start}, {end}, {label}] in the text because the character span "
                           f"'{doc.text[start:end]}' does not align with token boundaries:\n\n{repr(sentence)}\n")
                    warnings.warn(msg)
                else:
                    remained_count += 1
                    print("Successfully added entity")
                    ents.append(span)
                    entity_counts[label] += 1
            filtered = filter_spans(ents) # THIS DOES THE TRICK
            doc.ents = filtered  # Set the entities for the Doc
            db.add(doc)  # Add the Doc to DocBin

    db.to_disk(output_path)  # Save the DocBin to disk
    print(f"Saved to {output_path}")
    return entity_counts

# Example Usage

# Output paths
train_output_path = Path("C:/Users/Wenjuan.Zhang/OneDrive - FDA/Documents/train_VAVERSLLMLLM_revision.spacy")
test_output_path = Path("C:/Users/Wenjuan.Zhang/OneDrive - FDA/Documents/valid_VAVERS_LLMLLM_revision.spacy")

# Convert and save to .spacy format
train_entity_counts = convert("en", train_data, train_output_path)
test_entity_counts = convert("en", test_data, test_output_path)

Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully

C:\Users\Wenjuan.Zhang\AppData\Local\Temp\1\ipykernel_34812\637637866.py:158: UserWarning: Skipping entity [92, 101, vax] in the text because the character span 'ENGERIX B' does not align with token boundaries:

'The regulatory authority reported that the events were possibly related to vaccination with ENGERIX B.'

  warnings.warn(msg)
C:\Users\Wenjuan.Zhang\AppData\Local\Temp\1\ipykernel_34812\637637866.py:145: UserWarning: Skipping text with overlapping entities: 'Two days later, the patient had pain and swelling under the armpit, he had swelling on the side of the body (same side as injection), and he stated that he "feels it moving when he walks".'
  warnings.warn(f"Skipping text with overlapping entities: {repr(sentence)}")
C:\Users\Wenjuan.Zhang\AppData\Local\Temp\1\ipykernel_34812\637637866.py:158: UserWarning: Skipping entity [117, 119, sym] in the text because the character span 'BM' does not align with token boundaries:

'Records reveal patient seen by PCP w/fever, nausea & 

Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully

C:\Users\Wenjuan.Zhang\AppData\Local\Temp\1\ipykernel_34812\637637866.py:145: UserWarning: Skipping text with overlapping entities: "The patient's date of birth and complete expiration date of the vaccination were unknown."
  warnings.warn(f"Skipping text with overlapping entities: {repr(sentence)}")
C:\Users\Wenjuan.Zhang\AppData\Local\Temp\1\ipykernel_34812\637637866.py:145: UserWarning: Skipping text with overlapping entities: 'Previous and/or concurrent vaccination included bacillus calmette-guerin vaccine (non-gsk); non-GSK manufacturer; intradermal given on 23 May 2008; DTPa-IPV-HIB; manufacturer unspecified; intramuscular given on 21 July 2008 and 22 September 2008; hepatitis B vaccine recombinant; manufacturer unspecified; intramuscular given on 23 May 2008 and 21 August 2008; ROTARIX; GlaxoSmithKline; oral given on 21 July 28; pneumococcal vaccines (non-gsk); manufacturer unspecified; intramuscular given on 21 August 2008.'
  warnings.warn(f"Skipping text with overlapping enti

Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully

...

C:\Users\Wenjuan.Zhang\AppData\Local\Temp\1\ipykernel_34812\637637866.py:145: UserWarning: Skipping text with overlapping entities: "The patient received concomitant treatment with NEXIUM 20 mg orally daily since 05-Dec-2011 for Barrett's oesophagus, felodipine MR 10 mg orally daily since 08-Jan-2013 for hypertension, clopidogrel 75 mg orally daily since 20-May-2010, AMIAS 16 mg orally daily since 05-Jun-1999 for hypertension, tadalafil 5 mg orally daily since 20-May-2009 for erectile disorder and LIPOSTAT from 11-Oct-2011 for ischemic heart disease and all concomitant were still ongoing."
  warnings.warn(f"Skipping text with overlapping entities: {repr(sentence)}")
C:\Users\Wenjuan.Zhang\AppData\Local\Temp\1\ipykernel_34812\637637866.py:145: UserWarning: Skipping text with overlapping entities: 'The rest of her current rash was actually post inflammatory hyperpigmentation which is very commonly seen in skin phototype 4 (and above) after an inflammatory rash.'
  warnings.warn(f"Skippin

Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully

C:\Users\Wenjuan.Zhang\AppData\Local\Temp\1\ipykernel_34812\637637866.py:145: UserWarning: Skipping text with overlapping entities: 'While the hepatitis B vaccine was very effective at preventing chronic HBV infection, recent studies indicated it was less effective at preventing occult HBV following infant vaccination.'
  warnings.warn(f"Skipping text with overlapping entities: {repr(sentence)}")
C:\Users\Wenjuan.Zhang\AppData\Local\Temp\1\ipykernel_34812\637637866.py:145: UserWarning: Skipping text with overlapping entities: 'No studies, however, had examined the efficacy of adult HBV vaccination at preventing occult HBV.'
  warnings.warn(f"Skipping text with overlapping entities: {repr(sentence)}")
C:\Users\Wenjuan.Zhang\AppData\Local\Temp\1\ipykernel_34812\637637866.py:145: UserWarning: Skipping text with overlapping entities: 'Conclusions: HBV vaccination in infancy was effective at preventing chronic HBV infection but was less effective at preventing occult HBV infection.'
  warni

Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully added entity
Successfully

C:\Users\Wenjuan.Zhang\AppData\Local\Temp\1\ipykernel_34812\637637866.py:158: UserWarning: Skipping entity [116, 125, vax] in the text because the character span 'Engerix B' does not align with token boundaries:

'Follow-up 1 received on 7 April 2008: This case was reported via-e-mail by the subject who received the 3rd dose of Engerix B. Ten days after vaccination, he developed unspecified neurological disorder with pain.'

  warnings.warn(msg)
C:\Users\Wenjuan.Zhang\AppData\Local\Temp\1\ipykernel_34812\637637866.py:145: UserWarning: Skipping text with overlapping entities: 'Information has been received from a nurse, for GARDASIL, a Pregnancy Registry product, concerning a 20 year old female patient with asthma, migraines, depression, irregular heartbeat, high blood pressure, penicillin allergy and diphenhydramine allergy and a childhood history of anemia who on 29-SEP-2009 was vaccinated IM with her first dose of GARDASIL (lot # 663452/0671Y, expiration date 18-SEP-2011).'
  warning

In [58]:
train_data

[('This case was reported by a pharmacist and described the occurrence of injection site cellulitis in a 35-year-old female subject who was vaccinated with influenza virus vaccine (Fluarix), plus a separate injection of 23-valent pneumococcal polysaccharide vaccine (Pneumovax) for prophylaxis.',
  {'entities': [(71, 96, 'sym'),
    (102, 104, 'age'),
    (114, 120, 'sex'),
    (153, 186, 'vax'),
    (217, 274, 'vax')]}),
 ('Medical history included cellulitis following prior receipt of concomitant administration of influenza virus vaccine and Pneumovax and sickle cell anemia.',
  {'entities': [(25, 35, 'hx'),
    (93, 116, 'vax'),
    (121, 130, 'vax'),
    (135, 153, 'hx')]}),
 ('On 30 October 2006 at 9:00 a.m., the subject received a dose of Fluarix in the right thigh.',
  {'entities': [(3, 18, 'tempo'),
    (22, 31, 'tempo'),
    (56, 60, 'dose'),
    (64, 71, 'vax')]}),
 ('On 30 October 2006 the subject also received a dose of Pneumovax at an unspecified site of injection.',
  {'en

In [59]:
# Print the counts
print("Train Entity Counts:")
for entity_type, count in train_entity_counts.items():
    print(f"{entity_type}: {count}")

print("\nTest Entity Counts:")
for entity_type, count in test_entity_counts.items():
    print(f"{entity_type}: {count}")

Train Entity Counts:
sym: 7925
age: 1000
sex: 632
vax: 3368
hx: 996
tempo: 5700
dose: 645
dx: 1863
status: 329
tx: 2403
treatment: 289
ro: 231
lab: 2972

Test Entity Counts:
sex: 151
sym: 1976
vax: 803
hx: 221
dx: 492
tempo: 1380
dose: 140
tx: 521
age: 235
lab: 723
status: 93
ro: 63
treatment: 64


: 